# Video Chef -- Text-to-Video (Wan 2.2)

Generate a short video from a text prompt using the open-source **Wan 2.2** family (Apache 2.0).

**Default model: `TI2V-5B`** -- runs on a single **L4 (24 GB)**.
For higher quality, switch to `T2V-A14B` (requires A100 40 GB).

| Model | Task | Min VRAM | Colab tier | Speed (5 s @ 720p) |
|---|---|---|---|---|
| `ti2v-5B` | T2V + I2V | ~24 GB | L4 | ~9 min |
| `t2v-A14B` | T2V (MoE 27B/14B active) | ~40 GB | A100 40 GB | ~15-25 min |

> Runtime > Change runtime type > **L4** (or A100 for A14B).


In [ ]:
# @title 0. Check GPU
!nvidia-smi

In [ ]:
# @title 1. Mount Google Drive (shared model cache)
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/Wan2.2/outputs
print('Drive mounted. Shared weights root: /content/drive/MyDrive/Wan2.2/')


In [ ]:
# @title 2. Clone Wan 2.2 repo + install dependencies
%cd /content
![ -d Wan2.2 ] || git clone --depth 1 https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2
!pip install -q -e .
!pip install -q -r requirements.txt
!pip install -q ftfy regex decord loguru 'huggingface_hub[cli]'
print('Setup done.')


In [ ]:
# @title 3a. Pick model
# @markdown Pick the model. TI2V-5B = L4-friendly. T2V-A14B = A100 only.
MODEL = "TI2V-5B"  # @param ["TI2V-5B", "T2V-A14B"]


In [ ]:
# @title 3b. Download model weights (shared on Drive across notebooks)
import os, shutil, glob
from huggingface_hub import snapshot_download

REPO_ID  = f"Wan-AI/Wan2.2-{MODEL}"
CKPT_DIR = f"/content/drive/MyDrive/Wan2.2/Wan2.2-{MODEL}"

os.makedirs(CKPT_DIR, exist_ok=True)

# --- Storage check ---
_, _, free = shutil.disk_usage("/")
print(f"Local disk: {free // (2**30)} GB free")

# --- Check if model already exists on Drive ---
existing_files = os.listdir(CKPT_DIR) if os.path.isdir(CKPT_DIR) else []
print(f"Files already in {CKPT_DIR}: {len(existing_files)}")
for f in sorted(existing_files):
    full = os.path.join(CKPT_DIR, f)
    if os.path.isdir(full):
        sub_count = len(os.listdir(full))
        print(f"  [DIR]  {f}/  ({sub_count} files)")
    else:
        sz = os.path.getsize(full) / (1024**2)
        print(f"  [FILE] {f}  ({sz:.1f} MB)")

# --- Detect model weights: look for .safetensors or .bin in any subfolder ---
weight_files = (
    glob.glob(os.path.join(CKPT_DIR, "**", "*.safetensors"), recursive=True)
    + glob.glob(os.path.join(CKPT_DIR, "**", "*.bin"), recursive=True)
)
print(f"Weight files found: {len(weight_files)}")

if len(weight_files) >= 1:
    print(f"\u2705 Model weights found on Drive at {CKPT_DIR}. Skipping download.")
else:
    print(f"\u274c No model weights found on Drive. Downloading {REPO_ID} ...")
    snapshot_download(
        repo_id=REPO_ID,
        local_dir=CKPT_DIR,
    )
    print("Download complete.")

print(f"\nModel path for inference: {CKPT_DIR}")


In [ ]:
# @title 4. Prompt & settings
# @markdown ### Prompt and generation settings
PROMPT = "A cinematic shot of a fox running through a snowy forest at golden hour, 4k, shallow depth of field"  # @param {type:"string"}
SIZE   = "1280*704"  # @param ["1280*704", "704*1280", "832*480", "480*832"]
SEED   = 42  # @param {type:"integer"}
# @markdown ### Duration (frame count must be 4k+1, rendered at 16fps)
FRAME_NUM = 81  # @param [33, 49, 65, 81, 97, 113, 129] {type:"raw"}
# @markdown Leave IMAGE empty for pure text-to-video. Provide a path for image-to-video (TI2V-5B only).
IMAGE  = ""  # @param {type:"string"}
print(f"Duration: ~{FRAME_NUM/16:.1f}s ({FRAME_NUM} frames @ 16fps)")
print(f"MODEL={MODEL}  SIZE={SIZE}  SEED={SEED}  FRAMES={FRAME_NUM}")
print(f"PROMPT={PROMPT}")


In [ ]:
# @title 5. Run inference
import time
%cd /content/Wan2.2

if MODEL == "TI2V-5B":
    task  = "ti2v-5B"
    flags = "--offload_model True --convert_model_dtype --t5_cpu"
else:
    task  = "t2v-A14B"
    flags = "--offload_model True --convert_model_dtype"

img_arg = f'--image "{IMAGE}"' if IMAGE else ""
cmd = (
    f'python generate.py --task {task} --size {SIZE} '
    f'--ckpt_dir "{CKPT_DIR}" --base_seed {SEED} --frame_num {FRAME_NUM} {flags} {img_arg} '
    f'--prompt "{PROMPT}"'
)
print("Running:\n", cmd, "\n")
t0 = time.time()
!{cmd}
print(f"\nElapsed: {(time.time()-t0)/60:.1f} min")


In [ ]:
# @title 6. Show result + save to Drive
import glob, shutil, os, time
from IPython.display import HTML, display
from base64 import b64encode

vids = sorted(glob.glob("/content/Wan2.2/*.mp4"), key=os.path.getmtime, reverse=True)
assert vids, "No mp4 produced - check the inference cell output above."
latest = vids[0]

# Copy to Drive for safekeeping
ts = time.strftime("%Y%m%d_%H%M%S")
drive_out = f"/content/drive/MyDrive/Wan2.2/outputs/t2v_{MODEL}_{ts}.mp4"
shutil.copy(latest, drive_out)
print(f"Saved: {drive_out}")

data_url = "data:video/mp4;base64," + b64encode(open(latest, 'rb').read()).decode()
display(HTML(f'<video width=720 controls src="{data_url}"></video>'))
